# Treino dos modelos de saúde fetal

O fluxo `trees` usa GridSearchCV aninhada para Random Forest e XGBoost. O fluxo `cnn` treina a CNN 1D com TensorFlow/Keras. Ambos usam o split persistido e registram o refit no TRAIN completo. Consulte `03_model_metrics.ipynb` para as árvores ou `notebooks_metrics_dl.ipynb` para a CNN.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from saude_fetal.config import load_config

RUN_TRAINING = False
TRACKING_BACKEND = "local"  # Use "dagshub" para registrar no repositório remoto.
WORKFLOW = "trees"  # Use "cnn" para a CNN ou "all" para os dois fluxos.
print(load_config("model")["models"])

{'random_forest': {'estimator': 'random_forest', 'params': {'n_estimators': 150, 'n_jobs': 1}, 'grid': {'max_depth': [5, None], 'min_samples_leaf': [2, 8], 'class_weight': [None, 'balanced']}}, 'xgboost': {'estimator': 'xgboost', 'params': {'objective': 'multi:softprob', 'eval_metric': 'mlogloss', 'tree_method': 'hist', 'n_jobs': 1}, 'grid': {'n_estimators': [100, 200], 'max_depth': [2, 4], 'learning_rate': [0.05, 0.15]}}}


In [2]:
if RUN_TRAINING:
    from saude_fetal.train import run_training

    results = run_training(tracking_backend=TRACKING_BACKEND, workflow=WORKFLOW)
    display(results[["model", "cv_f1_macro_mean", "cv_f1_macro_std",
                     "cv_recall_patologico_mean", "run_id"]])
else:
    print("RUN_TRAINING=False. Consulte 03_model_metrics ou notebooks_metrics_dl.")

RUN_TRAINING=False. Consulte 03_model_metrics ou notebooks_metrics_dl.


Para consultar o tracking local, use `mlflow ui --backend-store-uri ./mlruns` na raiz do projeto. Com DagsHub, abra **Experiments** no [repositório](https://dagshub.com/mateusssssss/Saude-Fetal-pos-graduacao).